In [ ]:
from __future__ import annotations
import datetime as dt
import os
import random
import re
import warnings
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Mapping, Optional, Sequence, Tuple
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import distance_transform_edt
from scipy.stats import mannwhitneyu
from tqdm import tqdm
from project_config import CATALOG_ROOT, OUTPUT_ROOT, STATISTICS_ROOT, RANDOM_SEED, load_pickle
from pathlib import Path
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
RANDOM_SEED = 20260724
DEFAULT_YEAR = 2024
DATA_SAVE_ROOT = CATALOG_ROOT
OUTPUT_DIR = STATISTICS_ROOT
ALL_SPLITS = ('train', 'validation', 'test')
ANALYSIS_SPLITS = ('train', 'validation', 'test')
RADAR_KEYS = ['CR', 'VIL', 'EB20', 'ET20', 'EB30', 'ET30', 'EB45', 'ET45']
NWP_KEYS = ['HT0', 'HT10', 'HT20', 'HTw0']
ALL_PARAMS = RADAR_KEYS + NWP_KEYS
FRAME_INTERVAL_MIN = 6
N_PRE_FRAMES = 10
WINDOW_SIZE = 31
NEGATIVE_RATIO = 3
CR_CONVECTION_THRESHOLD = 30.0
MIN_VALID_FRACTION = 0.8
RADAR_NEGATIVE_FILL = 0.0
PRIMARY_STAT_MAP = {**{k: 'p90' for k in RADAR_KEYS}, **{k: 'median' for k in NWP_KEYS}}
TOP_RADAR_PARAMS = ['CR', 'VIL', 'EB45', 'ET45']
N_BOOT = 1000
N_PERM = 999
PARAM_LABELS = {'CR': 'Composite reflectivity', 'VIL': 'Vertically integrated liquid', 'EB20': '20-dBZ echo-base height', 'ET20': '20-dBZ echo-top height', 'EB30': '30-dBZ echo-base height', 'ET30': '30-dBZ echo-top height', 'EB45': '45-dBZ echo-base height', 'ET45': '45-dBZ echo-top height', 'HT0': '0°C level height', 'HT10': '−10°C level height', 'HT20': '−20°C level height', 'HTw0': 'Wet-bulb 0°C level height'}
PARAM_UNITS = {'CR': 'dBZ', 'VIL': 'kg m$^{-2}$', 'EB20': 'km', 'ET20': 'km', 'EB30': 'km', 'ET30': 'km', 'EB45': 'km', 'ET45': 'km', 'HT0': 'km', 'HT10': 'km', 'HT20': 'km', 'HTw0': 'km'}


In [ ]:
def set_reproducibility(seed: int=RANDOM_SEED) -> np.random.Generator:
    random.seed(seed)
    np.random.seed(seed)
    return np.random.default_rng(seed)

def normalize_path(path_like: str | os.PathLike) -> str:
    return str(path_like).replace('\\', '/')

def parse_data_time(path_like: str | os.PathLike, default_year: int=DEFAULT_YEAR) -> dt.datetime:
    stem = Path(normalize_path(path_like)).stem
    m = re.search('(?<!\\d)(20\\d{2})(\\d{2})(\\d{2})[-_](\\d{2})(\\d{2})(?!\\d)', stem)
    if m:
        year, month, day, hour, minute = map(int, m.groups())
        return dt.datetime(year, month, day, hour, minute)
    m = re.search('(?<!\\d)(\\d{2})(\\d{2})[-_](\\d{2})(\\d{2})(?!\\d)', stem)
    if m:
        month, day, hour, minute = map(int, m.groups())
        return dt.datetime(default_year, month, day, hour, minute)
    raise ValueError(f'Cannot parse timestamp from filename: {path_like}')

def parse_station_id_from_path(path_like: str | os.PathLike) -> str:
    parts = Path(normalize_path(path_like)).stem.split('_')
    if len(parts) < 3:
        raise ValueError(f'Cannot parse radar ID from filename: {path_like}')
    return parts[-3]

def load_extent_dict(doc_root: Path) -> Dict[str, List[float]]:
    extent_dict: Dict[str, List[float]] = {}
    for file_path in sorted(doc_root.iterdir()):
        if not file_path.is_file():
            continue
        parts = file_path.stem.split('_')
        if len(parts) < 3:
            continue
        station_id = parts[2]
        table = pd.read_csv(file_path)
        try:
            lon_start = round(float(str(table.iloc[0, 0]).split('=')[1]), 4)
            lat_start = round(float(str(table.iloc[1, 0]).split('=')[1]), 4)
            lon_end = round(float(str(table.iloc[2, 0]).split('=')[1]), 4)
            lat_end = round(float(str(table.iloc[3, 0]).split('=')[1]), 4)
        except (IndexError, ValueError) as exc:
            raise ValueError(f'Invalid extent-file format: {file_path}') from exc
        extent_dict[station_id] = [lon_start, lon_end, lat_start, lat_end]
    return extent_dict

@lru_cache(maxsize=1024)
def load_npy_cached(path: str) -> np.ndarray:
    arr = np.load(path)
    if arr.ndim != 2:
        raise ValueError(f'Only two-dimensional arrays are supported; shape={arr.shape}, file={path}')
    return np.asarray(arr, dtype=np.float64)

def fill_nan_nearest(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float64)
    nan_mask = ~np.isfinite(arr)
    if not np.any(nan_mask):
        return arr.copy()
    if np.all(nan_mask):
        return np.full_like(arr, np.nan)
    indices = distance_transform_edt(nan_mask, return_distances=False, return_indices=True)
    return arr[tuple(indices)]

def resize_linear(arr: np.ndarray, new_shape: Tuple[int, int]) -> np.ndarray:
    h_old, w_old = arr.shape
    h_new, w_new = new_shape
    y_old = np.arange(h_old)
    x_old = np.arange(w_old)
    y_new = np.linspace(0, h_old - 1, h_new)
    x_new = np.linspace(0, w_old - 1, w_new)
    yy, xx = np.meshgrid(y_new, x_new, indexing='ij')
    interpolator = RegularGridInterpolator((y_old, x_old), arr, method='linear', bounds_error=False, fill_value=np.nan)
    points = np.stack([yy, xx], axis=-1)
    return interpolator(points)

def label_frames_to_first_onset(label_frames: Sequence) -> Dict[Tuple[int, int], int]:
    first_onset: Dict[Tuple[int, int], int] = {}
    for frame_idx, frame_labels in enumerate(label_frames):
        for coord in frame_labels:
            x, y = (int(coord[0]), int(coord[1]))
            first_onset.setdefault((x, y), frame_idx)
    return first_onset

def normalize_station_locations(st_locs: Mapping) -> Dict[str, Tuple[int, int]]:
    out: Dict[str, Tuple[int, int]] = {}
    for station_id, value in st_locs.items():
        if value is None or len(value) < 2:
            continue
        out[str(station_id)] = (int(value[0]), int(value[1]))
    return out

def load_case_catalog(data_save_root: Path=DATA_SAVE_ROOT, splits: Sequence[str]=ALL_SPLITS) -> Dict[str, dict]:
    cases: Dict[str, dict] = {}
    for split in splits:
        split_root = data_save_root / split
        data_dic = load_pickle(split_root / 'data_dic_nwp.pkl')
        mask_dic_all = load_pickle(split_root / 'mask_dic_all.pkl')
        for case_id, raw_case in data_dic.items():
            case_uid = f'{split}::{case_id}'
            case = {'case_uid': case_uid, 'case_id': case_id, 'split': split, 'label': label_frames_to_first_onset(raw_case.get('label', [])), 'st_loc': normalize_station_locations(mask_dic_all.get(case_id, {}))}
            for key in ALL_PARAMS:
                if key in raw_case:
                    case[key] = list(raw_case[key])
            cases[case_uid] = case
    return cases

def get_radar(case: Mapping, key: str, frame_idx: int) -> Tuple[np.ndarray, dt.datetime, str]:
    if key not in RADAR_KEYS:
        raise KeyError(f'Unknown radar parameter: {key}')
    paths = case.get(key, [])
    if frame_idx < 0 or frame_idx >= len(paths):
        raise IndexError(f"{case['case_uid']} {key} frame={frame_idx} is out of bounds")
    path = str(paths[frame_idx])
    data = load_npy_cached(path).copy()
    data[data < 0] = RADAR_NEGATIVE_FILL
    valid_time = parse_data_time(path)
    return (data, valid_time, path)

def choose_nwp_path(case: Mapping, key: str, radar_time: dt.datetime, max_age_minutes: int=120) -> Tuple[str, dt.datetime]:
    if key not in NWP_KEYS:
        raise KeyError(f'Unknown NWP parameter: {key}')
    candidates = []
    for path in case.get(key, []):
        try:
            valid_time = parse_data_time(path)
        except ValueError:
            continue
        if valid_time <= radar_time:
            candidates.append((valid_time, str(path)))
    if not candidates:
        raise FileNotFoundError(f"{case['case_uid']} {key}: radar_time={radar_time}: no earlier NWP field is available")
    valid_time, path = max(candidates, key=lambda x: x[0])
    age_min = (radar_time - valid_time).total_seconds() / 60.0
    if age_min > max_age_minutes:
        warnings.warn(f"{case['case_uid']} {key}: NWP field age {age_min:.0f} min exceeds {max_age_minutes} min")
    return (path, valid_time)

def get_nwp(case: Mapping, key: str, radar_time: dt.datetime, radar_extent: Mapping[str, Sequence[float]], nwp_extent: Mapping[str, Sequence[float]]) -> Tuple[np.ndarray, dt.datetime, str]:
    path, valid_time = choose_nwp_path(case, key, radar_time)
    station_id = parse_station_id_from_path(path)
    data = fill_nan_nearest(load_npy_cached(path))
    data = resize_linear(data, (501, 501))
    if station_id not in radar_extent or station_id not in nwp_extent:
        raise KeyError(f'Station {station_id} is missing radar or NWP geographic extents')
    start_y = int(round(abs(nwp_extent[station_id][2] - radar_extent[station_id][2]) / 0.01))
    start_x = int(round(abs(nwp_extent[station_id][0] - radar_extent[station_id][0]) / 0.01))
    crop = data[start_y:start_y + 401, start_x:start_x + 401]
    if crop.shape != (401, 401):
        raise ValueError(f'Invalid NWP crop: station={station_id}, shape={crop.shape}, path={path}')
    return (crop, valid_time, path)

def get_window_values(data_2d: np.ndarray, x: int, y: int, window_size: int=WINDOW_SIZE) -> np.ndarray:
    half = window_size // 2
    h, w = data_2d.shape
    y0, y1 = (max(0, y - half), min(h, y + half + 1))
    x0, x1 = (max(0, x - half), min(w, x + half + 1))
    return np.asarray(data_2d[y0:y1, x0:x1], dtype=np.float64)

def get_window_stats(data_2d: np.ndarray, x: int, y: int, window_size: int=WINDOW_SIZE, threshold: Optional[float]=None) -> Dict[str, float]:
    window = get_window_values(data_2d, x, y, window_size)
    finite = window[np.isfinite(window)]
    valid_fraction = finite.size / max(window.size, 1)
    keys = ['mean', 'median', 'max', 'p90', 'p95', 'iqr', 'std', 'valid_fraction']
    if finite.size == 0 or valid_fraction < MIN_VALID_FRACTION:
        out = {k: np.nan for k in keys}
        out['valid_fraction'] = valid_fraction
        if threshold is not None:
            out['fraction_ge_threshold'] = np.nan
        return out
    q25, q75 = np.percentile(finite, [25, 75])
    out = {'mean': float(np.mean(finite)), 'median': float(np.median(finite)), 'max': float(np.max(finite)), 'p90': float(np.percentile(finite, 90)), 'p95': float(np.percentile(finite, 95)), 'iqr': float(q75 - q25), 'std': float(np.std(finite, ddof=1)) if finite.size > 1 else 0.0, 'valid_fraction': float(valid_fraction)}
    if threshold is not None:
        out['fraction_ge_threshold'] = float(np.mean(finite >= threshold))
    return out

def has_strong_convection(case: Mapping, x: int, y: int, onset_frame: int) -> bool:
    for offset in range(N_PRE_FRAMES, 0, -1):
        frame_idx = onset_frame - offset
        try:
            data, _, _ = get_radar(case, 'CR', frame_idx)
        except Exception:
            return False
        stats = get_window_stats(data, x, y, threshold=CR_CONVECTION_THRESHOLD)
        if np.isfinite(stats['max']) and stats['max'] >= CR_CONVECTION_THRESHOLD:
            return True
    return False

def build_matched_samples(cases: Mapping[str, Mapping], splits: Sequence[str]=ANALYSIS_SPLITS, negative_ratio: int=NEGATIVE_RATIO, seed: int=RANDOM_SEED) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    samples: List[dict] = []
    issues: List[dict] = []
    for case_uid, case in tqdm(cases.items(), desc='Matching samples', unit='case'):
        if case['split'] not in splits:
            continue
        hail_onsets: Dict[Tuple[int, int], int] = case.get('label', {})
        station_locs: Dict[str, Tuple[int, int]] = case.get('st_loc', {})
        valid_pos = [(coord, frame) for coord, frame in hail_onsets.items() if frame >= N_PRE_FRAMES]
        hail_coords = set(hail_onsets.keys())
        if not valid_pos:
            issues.append({'case_uid': case_uid, 'reason': 'no_positive_with_full_history'})
            continue
        base_neg_candidates = [(station_id, coord) for station_id, coord in station_locs.items() if coord not in hail_coords]
        for pos_idx, ((x, y), onset_frame) in enumerate(valid_pos):
            pos_id = f'{case_uid}::H::{x}_{y}::{onset_frame}'
            samples.append({'sample_id': pos_id, 'case_uid': case_uid, 'case_id': case['case_id'], 'split': case['split'], 'label': 'Hail', 'label_binary': 1, 'station_id': f'hail_xy_{x}_{y}', 'x': x, 'y': y, 'onset_frame': onset_frame, 'matched_positive_id': pos_id})
            order = rng.permutation(len(base_neg_candidates))
            n_added = 0
            for idx in order:
                station_id, (nx, ny) = base_neg_candidates[idx]
                if not has_strong_convection(case, nx, ny, onset_frame):
                    continue
                neg_id = f'{case_uid}::N::{station_id}::{onset_frame}'
                samples.append({'sample_id': neg_id, 'case_uid': case_uid, 'case_id': case['case_id'], 'split': case['split'], 'label': 'No hail (convection)', 'label_binary': 0, 'station_id': station_id, 'x': nx, 'y': ny, 'onset_frame': onset_frame, 'matched_positive_id': pos_id})
                n_added += 1
                if n_added >= negative_ratio:
                    break
            if n_added < negative_ratio:
                issues.append({'case_uid': case_uid, 'sample_id': pos_id, 'reason': 'insufficient_convective_controls', 'n_controls': n_added})
    return (pd.DataFrame(samples), pd.DataFrame(issues))

def extract_statistics(cases: Mapping[str, Mapping], samples: pd.DataFrame, radar_extent: Mapping[str, Sequence[float]], nwp_extent: Mapping[str, Sequence[float]]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    records: List[dict] = []
    issues: List[dict] = []
    for row in tqdm(samples.itertuples(index=False), total=len(samples), desc='Extracting statistics', unit='trajectory'):
        case = cases[row.case_uid]
        try:
            _, hail_time, _ = get_radar(case, 'CR', int(row.onset_frame))
        except Exception as exc:
            issues.append({'sample_id': row.sample_id, 'reason': 'hail_time_read_failed', 'error': repr(exc)})
            continue
        for offset in range(N_PRE_FRAMES, 0, -1):
            frame_idx = int(row.onset_frame) - offset
            nominal_lead = -offset * FRAME_INTERVAL_MIN
            for param in RADAR_KEYS:
                try:
                    data, valid_time, source_path = get_radar(case, param, frame_idx)
                    threshold = CR_CONVECTION_THRESHOLD if param == 'CR' else None
                    stats = get_window_stats(data, int(row.x), int(row.y), threshold=threshold)
                except Exception as exc:
                    issues.append({'sample_id': row.sample_id, 'param': param, 'frame_idx': frame_idx, 'reason': 'radar_read_failed', 'error': repr(exc)})
                    continue
                for stat_name, value in stats.items():
                    records.append({**row._asdict(), 'source': 'radar', 'param': param, 'stat': stat_name, 'value': value, 'lead_min': nominal_lead, 'source_valid_time': valid_time, 'source_path': source_path, 'hail_time': hail_time})
            try:
                _, radar_time, _ = get_radar(case, 'CR', frame_idx)
            except Exception:
                continue
            for param in NWP_KEYS:
                try:
                    data, valid_time, source_path = get_nwp(case, param, radar_time, radar_extent, nwp_extent)
                    stats = get_window_stats(data, int(row.x), int(row.y))
                except Exception as exc:
                    issues.append({'sample_id': row.sample_id, 'param': param, 'frame_idx': frame_idx, 'reason': 'nwp_read_failed', 'error': repr(exc)})
                    continue
                native_lead = int(round((valid_time - hail_time).total_seconds() / 60.0))
                for stat_name, value in stats.items():
                    records.append({**row._asdict(), 'source': 'nwp', 'param': param, 'stat': stat_name, 'value': value, 'lead_min': native_lead, 'source_valid_time': valid_time, 'source_path': source_path, 'hail_time': hail_time})
    df = pd.DataFrame(records)
    issues_df = pd.DataFrame(issues)
    if df.empty:
        return (df, issues_df)
    radar_df = df[df['source'] == 'radar'].copy()
    radar_df['native_lead_min'] = radar_df['lead_min']
    nwp_df = df[df['source'] == 'nwp'].sort_values('source_valid_time').drop_duplicates(subset=['sample_id', 'param', 'stat', 'source_valid_time'], keep='last').copy()
    nwp_df['native_lead_min'] = nwp_df['lead_min']
    nwp_df = nwp_df[(nwp_df['native_lead_min'] >= -60) & (nwp_df['native_lead_min'] < 0)]
    if not nwp_df.empty:
        idx = nwp_df.groupby(['sample_id', 'param', 'stat'])['source_valid_time'].idxmax()
        nwp_df = nwp_df.loc[idx].copy()
        nwp_df['lead_min'] = 0
    tidy = pd.concat([radar_df, nwp_df], ignore_index=True)
    return (tidy, issues_df)

def signed_auc(hail_values: np.ndarray, nohail_values: np.ndarray) -> float:
    hail = np.asarray(hail_values, dtype=float)
    nohail = np.asarray(nohail_values, dtype=float)
    hail = hail[np.isfinite(hail)]
    nohail = nohail[np.isfinite(nohail)]
    if hail.size == 0 or nohail.size == 0:
        return np.nan
    u = mannwhitneyu(hail, nohail, alternative='two-sided').statistic
    auc = u / (hail.size * nohail.size)
    return float(2.0 * auc - 1.0)

def bh_fdr(p_values: Sequence[float]) -> np.ndarray:
    p = np.asarray(p_values, dtype=float)
    q = np.full_like(p, np.nan)
    valid = np.isfinite(p)
    if not np.any(valid):
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    qv = np.empty_like(adjusted)
    qv[order] = adjusted
    q[valid] = qv
    return q

def cluster_bootstrap_signed_auc(sub: pd.DataFrame, n_boot: int, rng: np.random.Generator) -> Tuple[float, float]:
    case_ids = sub['case_uid'].dropna().unique()
    if len(case_ids) < 2:
        return (np.nan, np.nan)
    boot_values = []
    grouped = {cid: g for cid, g in sub.groupby('case_uid', sort=False)}
    for _ in range(n_boot):
        sampled_cases = rng.choice(case_ids, size=len(case_ids), replace=True)
        pieces = [grouped[cid] for cid in sampled_cases]
        boot = pd.concat(pieces, ignore_index=True)
        h = boot.loc[boot['label_binary'] == 1, 'value'].to_numpy()
        n = boot.loc[boot['label_binary'] == 0, 'value'].to_numpy()
        effect = signed_auc(h, n)
        if np.isfinite(effect):
            boot_values.append(effect)
    if not boot_values:
        return (np.nan, np.nan)
    return tuple(np.percentile(boot_values, [2.5, 97.5]))

def case_stratified_permutation_p(sub: pd.DataFrame, observed: float, n_perm: int, rng: np.random.Generator) -> float:
    if not np.isfinite(observed):
        return np.nan
    work = sub[['case_uid', 'sample_id', 'label_binary', 'value']].dropna().copy()
    by_case = {cid: g.copy() for cid, g in work.groupby('case_uid', sort=False)}
    perm_effects = []
    for _ in range(n_perm):
        pieces = []
        for _, g in by_case.items():
            g2 = g.copy()
            labels = g2['label_binary'].to_numpy().copy()
            rng.shuffle(labels)
            g2['perm_label'] = labels
            pieces.append(g2)
        perm = pd.concat(pieces, ignore_index=True)
        h = perm.loc[perm['perm_label'] == 1, 'value'].to_numpy()
        n = perm.loc[perm['perm_label'] == 0, 'value'].to_numpy()
        effect = signed_auc(h, n)
        if np.isfinite(effect):
            perm_effects.append(effect)
    if not perm_effects:
        return np.nan
    perm_effects = np.asarray(perm_effects)
    return float((1 + np.sum(np.abs(perm_effects) >= abs(observed))) / (len(perm_effects) + 1))

def select_primary_statistics(tidy: pd.DataFrame) -> pd.DataFrame:
    mask = np.zeros(len(tidy), dtype=bool)
    for param, stat_name in PRIMARY_STAT_MAP.items():
        mask |= ((tidy['param'] == param) & (tidy['stat'] == stat_name)).to_numpy()
    return tidy.loc[mask].copy()

def compute_effect_table(tidy_primary: pd.DataFrame, n_boot: int=N_BOOT, n_perm: int=N_PERM, seed: int=RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    group_cols = ['source', 'param', 'stat', 'lead_min']
    for keys, sub in tqdm(tidy_primary.groupby(group_cols, sort=True), desc='Computing effect sizes', unit='cell'):
        source, param, stat_name, lead = keys
        sub = sub[np.isfinite(sub['value'])].copy()
        h = sub.loc[sub['label_binary'] == 1, 'value'].to_numpy()
        n = sub.loc[sub['label_binary'] == 0, 'value'].to_numpy()
        effect = signed_auc(h, n)
        ci_low, ci_high = cluster_bootstrap_signed_auc(sub, n_boot, rng)
        p_value = case_stratified_permutation_p(sub, effect, n_perm, rng)
        rows.append({'source': source, 'param': param, 'stat': stat_name, 'lead_min': int(lead), 'signed_auc': effect, 'ci_low': ci_low, 'ci_high': ci_high, 'p_value': p_value, 'n_hail': int(np.isfinite(h).sum()), 'n_nohail': int(np.isfinite(n).sum()), 'n_cases': int(sub['case_uid'].nunique()), 'median_native_lead_min': float(np.nanmedian(sub['native_lead_min'])) if 'native_lead_min' in sub and np.isfinite(sub['native_lead_min']).any() else np.nan, 'hail_median': float(np.nanmedian(h)) if h.size else np.nan, 'nohail_median': float(np.nanmedian(n)) if n.size else np.nan, 'hail_q25': float(np.nanpercentile(h, 25)) if h.size else np.nan, 'hail_q75': float(np.nanpercentile(h, 75)) if h.size else np.nan, 'nohail_q25': float(np.nanpercentile(n, 25)) if n.size else np.nan, 'nohail_q75': float(np.nanpercentile(n, 75)) if n.size else np.nan})
    effect_df = pd.DataFrame(rows)
    if not effect_df.empty:
        effect_df['q_value'] = bh_fdr(effect_df['p_value'].to_numpy())
    return effect_df

def latest_nwp_effects(effect_df: pd.DataFrame) -> pd.DataFrame:
    nwp = effect_df[(effect_df['source'] == 'nwp') & (effect_df['lead_min'] < 0)].copy()
    if nwp.empty:
        return nwp
    idx = nwp.groupby('param')['lead_min'].idxmax()
    return nwp.loc[idx].sort_values('signed_auc')

def trajectory_summary(tidy_primary: pd.DataFrame, param: str, n_boot: int=N_BOOT, seed: int=RANDOM_SEED) -> pd.DataFrame:
    sub = tidy_primary[(tidy_primary['source'] == 'radar') & (tidy_primary['param'] == param) & np.isfinite(tidy_primary['value'])].copy()
    case_level = sub.groupby(['case_uid', 'label', 'lead_min'], as_index=False)['value'].median()
    case_ids = case_level['case_uid'].unique()
    rng = np.random.default_rng(seed + sum(map(ord, param)))
    out = []
    for (label, lead), g in case_level.groupby(['label', 'lead_min'], sort=True):
        point = float(np.median(g['value']))
        boot_points = []
        if len(case_ids) >= 2:
            grouped = {cid: cg for cid, cg in case_level.groupby('case_uid', sort=False)}
            for _ in range(n_boot):
                sampled = rng.choice(case_ids, size=len(case_ids), replace=True)
                pieces = [grouped[cid] for cid in sampled]
                boot = pd.concat(pieces, ignore_index=True)
                vals = boot.loc[(boot['label'] == label) & (boot['lead_min'] == lead), 'value'].to_numpy()
                if vals.size:
                    boot_points.append(np.median(vals))
        if boot_points:
            low, high = np.percentile(boot_points, [2.5, 97.5])
        else:
            low, high = (np.nan, np.nan)
        out.append({'param': param, 'label': label, 'lead_min': int(lead), 'median': point, 'ci_low': low, 'ci_high': high, 'n_cases': int(g['case_uid'].nunique())})
    return pd.DataFrame(out)

def configure_publication_style() -> None:
    mpl.rcParams.update({'font.family': 'Arial', 'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7, 'axes.linewidth': 0.7, 'xtick.major.width': 0.7, 'ytick.major.width': 0.7, 'xtick.major.size': 3, 'ytick.major.size': 3, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.transparent': False})

def panel_label(ax: plt.Axes, text: str) -> None:
    ax.text(-0.12, 1.05, text, transform=ax.transAxes, ha='left', va='bottom', fontsize=9, fontweight='bold')

def plot_effect_heatmap(ax: plt.Axes, effect_df: pd.DataFrame, params: Sequence[str], title: str) -> None:
    sub = effect_df[effect_df['param'].isin(params)].copy()
    leads = sorted(sub['lead_min'].unique())
    matrix = sub.pivot(index='param', columns='lead_min', values='signed_auc').reindex(index=params, columns=leads)
    qmat = sub.pivot(index='param', columns='lead_min', values='q_value').reindex(index=params, columns=leads)
    image = ax.imshow(matrix.to_numpy(), cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto', interpolation='nearest')
    ax.set_yticks(np.arange(len(params)))
    ax.set_yticklabels([p for p in params])
    ax.set_xticks(np.arange(len(leads)))
    ax.set_xticklabels(leads)
    ax.set_xlabel('Time relative to hail onset (min)')
    ax.set_title(title, loc='left', fontweight='bold')
    for i in range(len(params)):
        for j in range(len(leads)):
            q = qmat.iloc[i, j]
            if np.isfinite(q) and q < 0.05:
                ax.plot(j, i, 'o', ms=2.2, color='black')
    for spine in ax.spines.values():
        spine.set_visible(False)
    return image

def plot_nwp_bars(ax: plt.Axes, latest_nwp: pd.DataFrame) -> None:
    if latest_nwp.empty:
        ax.text(0.5, 0.5, 'No independent NWP valid time\nwithin −60 to 0 min', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        return
    y = np.arange(len(latest_nwp))
    values = latest_nwp['signed_auc'].to_numpy()
    xerr = np.vstack([values - latest_nwp['ci_low'].to_numpy(), latest_nwp['ci_high'].to_numpy() - values])
    colors = np.where(values >= 0, '#D55E00', '#0072B2')
    ax.barh(y, values, xerr=xerr, color=colors, alpha=0.88, capsize=2, linewidth=0)
    ax.axvline(0, color='0.25', lw=0.7)
    ax.set_yticks(y)
    nwp_labels = []
    for row in latest_nwp.itertuples(index=False):
        native_lead = getattr(row, 'median_native_lead_min', np.nan)
        if np.isfinite(native_lead):
            nwp_labels.append(f'{row.param} ({native_lead:.0f} min)')
        else:
            nwp_labels.append(row.param)
    ax.set_yticklabels(nwp_labels)
    ax.set_xlim(-1, 1)
    ax.set_xlabel('Signed AUC')
    ax.set_title('Latest independent NWP field before hail', loc='left', fontweight='bold')
    ax.spines[['top', 'right', 'left']].set_visible(False)

def plot_trajectory(ax: plt.Axes, summary: pd.DataFrame, param: str) -> None:
    colors = {'Hail': '#D55E00', 'No hail (convection)': '#0072B2'}
    for label in ['Hail', 'No hail (convection)']:
        sub = summary[summary['label'] == label].sort_values('lead_min')
        if sub.empty:
            continue
        x = sub['lead_min'].to_numpy()
        y = sub['median'].to_numpy()
        low = sub['ci_low'].to_numpy()
        high = sub['ci_high'].to_numpy()
        ax.plot(x, y, marker='o', ms=2.6, lw=1.3, color=colors[label], label=label)
        ax.fill_between(x, low, high, color=colors[label], alpha=0.16, linewidth=0)
    ax.axvline(0, color='0.35', lw=0.7, ls='--')
    ax.set_xlim(-63, 2)
    ax.set_xticks([-60, -48, -36, -24, -12, 0])
    ax.set_xlabel('Time to hail onset (min)')
    unit = PARAM_UNITS.get(param, '')
    ylabel = f'{PRIMARY_STAT_MAP[param].upper()} ({unit})' if unit else PRIMARY_STAT_MAP[param].upper()
    ax.set_ylabel(ylabel)
    ax.set_title(f'{param}: {PARAM_LABELS.get(param, param)}', loc='left', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

def make_main_figure(tidy_primary: pd.DataFrame, effect_df: pd.DataFrame, output_dir: Path=OUTPUT_DIR, top_params: Sequence[str]=TOP_RADAR_PARAMS) -> plt.Figure:
    configure_publication_style()
    output_dir.mkdir(parents=True, exist_ok=True)
    fig = plt.figure(figsize=(7.2, 8.2))
    gs = fig.add_gridspec(3, 2, height_ratios=[1.45, 1.0, 1.0], hspace=0.58, wspace=0.42)
    ax_a = fig.add_subplot(gs[0, :])
    radar_effects = effect_df[effect_df['source'] == 'radar']
    image = plot_effect_heatmap(ax_a, radar_effects, RADAR_KEYS, 'Radar predictors: hail versus convective no-hail controls')
    panel_label(ax_a, 'A')
    cbar = fig.colorbar(image, ax=ax_a, fraction=0.025, pad=0.02)
    cbar.set_label('Signed AUC (hail higher →)')
    cbar.ax.tick_params(labelsize=7)
    ax_b = fig.add_subplot(gs[1, 0])
    plot_nwp_bars(ax_b, latest_nwp_effects(effect_df))
    panel_label(ax_b, 'B')
    trajectory_axes = [fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])]
    plotted_params = list(top_params)[:3]
    labels = ['C', 'D', 'E']
    for ax, param, label in zip(trajectory_axes, plotted_params, labels):
        summary = trajectory_summary(tidy_primary, param)
        plot_trajectory(ax, summary, param)
        panel_label(ax, label)
    handles, legend_labels = trajectory_axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, legend_labels, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=2, frameon=False)
    fig.subplots_adjust(bottom=0.08, left=0.12, right=0.97, top=0.97)
    fig.savefig(output_dir / 'Fig_pre_hail_statistics.pdf', bbox_inches='tight')
    fig.savefig(output_dir / 'Fig_pre_hail_statistics.tif', dpi=600, bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})
    fig.savefig(output_dir / 'Fig_pre_hail_statistics.png', dpi=300, bbox_inches='tight')
    return fig


In [ ]:
OUTPUT_DIR = STATISTICS_ROOT
RADAR_STAT = 'max'
NWP_STAT = 'max'
RADAR_Q_LOW = 0.25
RADAR_Q_HIGH = 0.75
NWP_BOX_WHIS = (5, 95)
RADAR_KEYS = ['CR', 'VIL', 'EB20', 'ET20', 'EB30', 'ET30', 'EB45', 'ET45']
NWP_KEYS = ['HT0', 'HT10', 'HT20', 'HTw0']
RADAR_LEADS = [-60, -54, -48, -42, -36, -30, -24, -18, -12, -6]
PARAM_UNITS = {'CR': 'dBZ', 'VIL': 'kg m$^{-2}$', 'EB20': 'km', 'ET20': 'km', 'EB30': 'km', 'ET30': 'km', 'EB45': 'km', 'ET45': 'km', 'HT0': 'm', 'HT10': 'm', 'HT20': 'm', 'HTw0': 'm'}
LABEL_ORDER = ['Hail', 'No hail']
LABEL_COLORS = {'Hail': '#D55E00', 'No hail': '#0072B2'}
A_EFFECT_VMAX: Optional[float] = 2
B_EFFECT_XLIM: Optional[Tuple[float, float]] = None
SIGNIFICANCE_LEVEL = 0.05
RADAR_YLIMS: Dict[str, Tuple[float, float]] = {}
NWP_YLIMS = {'HT0': (3000, 6000), 'HT10': (4500, 7800), 'HT20': (6000, 9500), 'HTw0': (3000, 6000)}
nwp_title_map = {'HT0': '$\\mathbf{H}_{\\mathbf{0}}$', 'HT10': '$\\mathbf{H}_{\\mathbf{-10}}$', 'HT20': '$\\mathbf{H}_{\\mathbf{-20}}$', 'HTw0': '$\\mathbf{H}_{\\mathbf{w0}}$'}
NWP_label = {'HT0': '$\\mathrm{H}_{0}$', 'HT10': '$\\mathrm{H}_{-10}$', 'HT20': '$\\mathrm{H}_{-20}$', 'HTw0': '$\\mathrm{H}_{\\mathrm{w0}}$'}
RADAR_LEGEND_Y_OFFSET = 0.04


In [ ]:
import string
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_saved_dataframe(stem: Path) -> pd.DataFrame:
    stem = Path(stem)
    parquet_path = Path(str(stem) + '.parquet')
    csv_path = Path(str(stem) + '.csv')
    if parquet_path.exists():
        try:
            print(f'Reading: {parquet_path}')
            return pd.read_parquet(parquet_path)
        except Exception as exc:
            print(f'Parquet read failed; trying CSV: {exc}')
    if csv_path.exists():
        print(f'Reading: {csv_path}')
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Files not found:\n  {parquet_path}\n  {csv_path}')

def select_primary_data(tidy: pd.DataFrame) -> pd.DataFrame:
    required = {'source', 'param', 'stat', 'value', 'lead_min', 'label_binary', 'case_uid'}
    missing = required.difference(tidy.columns)
    if missing:
        raise KeyError(f'tidy_stats is missing columns: {sorted(missing)}')
    radar_mask = tidy['source'].eq('radar') & tidy['param'].isin(RADAR_KEYS) & tidy['stat'].astype(str).str.lower().eq(RADAR_STAT.lower())
    nwp_mask = tidy['source'].eq('nwp') & tidy['param'].isin(NWP_KEYS) & tidy['stat'].astype(str).str.lower().eq(NWP_STAT.lower())
    primary = tidy.loc[radar_mask | nwp_mask].copy()
    primary['value'] = pd.to_numeric(primary['value'], errors='coerce')
    primary['lead_min'] = pd.to_numeric(primary['lead_min'], errors='coerce')
    primary['label_binary'] = pd.to_numeric(primary['label_binary'], errors='coerce')
    primary['label'] = np.where(primary['label_binary'].eq(1), 'Hail', 'No hail')
    if 'native_lead_min' in primary.columns:
        primary['native_lead_min'] = pd.to_numeric(primary['native_lead_min'], errors='coerce')
    primary = primary[np.isfinite(primary['value'])].copy()
    if primary.empty:
        raise RuntimeError('tidy_stats contains no data matching RADAR_STAT and NWP_STAT.')
    return primary

def load_paired_effect_table() -> pd.DataFrame:
    stem = OUTPUT_DIR / f'paired_robust_effect_table_{RADAR_STAT}_{NWP_STAT}'
    try:
        effect_df = load_saved_dataframe(stem)
    except FileNotFoundError as exc:
        raise FileNotFoundError(f'{exc}\n\nSupply the paired robust-effect table described in README.md; its generating script was not included in the source archive.') from exc
    required = {'source', 'param', 'stat', 'lead_min', 'paired_robust_effect', 'ci_low', 'ci_high', 'n_cases'}
    missing = required.difference(effect_df.columns)
    if missing:
        raise KeyError(f'Effect table is missing columns: {sorted(missing)}. Supply a compatible effect table.')
    effect_df = effect_df.copy()
    for column in ['lead_min', 'paired_robust_effect', 'ci_low', 'ci_high', 'n_cases']:
        effect_df[column] = pd.to_numeric(effect_df[column], errors='coerce')
    if 'q_value' in effect_df.columns:
        effect_df['q_value'] = pd.to_numeric(effect_df['q_value'], errors='coerce')
    return effect_df

def radar_distribution_summary(primary: pd.DataFrame, param: str, q_low: float=RADAR_Q_LOW, q_high: float=RADAR_Q_HIGH) -> pd.DataFrame:
    sub = primary[primary['source'].eq('radar') & primary['param'].eq(param) & np.isfinite(primary['value'])].copy()
    if sub.empty:
        return pd.DataFrame()
    case_level = sub.groupby(['case_uid', 'label', 'lead_min'], as_index=False)['value'].median()
    rows = []
    for (label, lead), group in case_level.groupby(['label', 'lead_min'], sort=True):
        values = group['value'].dropna().to_numpy(float)
        if values.size == 0:
            continue
        rows.append({'param': param, 'label': label, 'lead_min': int(lead), 'median': float(np.median(values)), 'q_low': float(np.quantile(values, q_low)), 'q_high': float(np.quantile(values, q_high)), 'n_cases': int(values.size)})
    return pd.DataFrame(rows)

def nwp_case_level_values(primary: pd.DataFrame, param: str) -> pd.DataFrame:
    sub = primary[primary['source'].eq('nwp') & primary['param'].eq(param) & np.isfinite(primary['value'])].copy()
    if sub.empty:
        return pd.DataFrame()
    return sub.groupby(['case_uid', 'label'], as_index=False)['value'].median()

def configure_style() -> None:
    mpl.rcParams.update({'font.family': 'Arial', 'font.size': 14, 'axes.labelsize': 14, 'axes.titlesize': 18, 'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 14, 'axes.linewidth': 1.0, 'xtick.major.width': 1.0, 'ytick.major.width': 1.0, 'xtick.major.size': 4, 'ytick.major.size': 4, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.transparent': False})

def add_panel_label(ax: plt.Axes, label: str) -> None:
    if label in {'A', 'B'}:
        x_position = -0.1
        y_position = 1.05
    else:
        x_position = -0.2
        y_position = 1.08
    ax.text(x_position, y_position, label, transform=ax.transAxes, ha='left', va='bottom', fontname='Arial', fontsize=23, fontweight='black')

def apply_axis_font_style(ax: plt.Axes) -> None:
    ax.title.set_fontname('Arial')
    ax.title.set_fontsize(18)
    ax.title.set_fontweight('bold')
    ax.xaxis.label.set_fontname('Arial')
    ax.xaxis.label.set_fontsize(14)
    ax.yaxis.label.set_fontname('Arial')
    ax.yaxis.label.set_fontsize(14)
    for tick_label in ax.get_xticklabels():
        tick_label.set_fontname('Arial')
        tick_label.set_fontsize(14)
    for tick_label in ax.get_yticklabels():
        tick_label.set_fontname('Arial')
        tick_label.set_fontsize(14)

def choose_effect_vmax(values: np.ndarray) -> float:
    if A_EFFECT_VMAX is not None:
        return float(A_EFFECT_VMAX)
    finite = np.abs(np.asarray(values, dtype=float))
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return 1.0
    vmax = float(np.percentile(finite, 95))
    vmax = max(vmax, 0.5)
    vmax = min(vmax, 8.0)
    return vmax

def plot_paired_effect_heatmap(ax: plt.Axes, radar_effects: pd.DataFrame) -> mpl.image.AxesImage:
    table = radar_effects.pivot_table(index='param', columns='lead_min', values='paired_robust_effect', aggfunc='median').reindex(index=RADAR_KEYS, columns=RADAR_LEADS)
    values = table.to_numpy(float)
    vmax = choose_effect_vmax(values)
    print(f'Panel A effect color limits: {-vmax:.3f}～{vmax:.3f}')
    image = ax.imshow(values, aspect='auto', origin='upper', cmap='RdBu_r', vmin=-vmax, vmax=vmax, interpolation='nearest')
    ax.set_xticks(np.arange(len(RADAR_LEADS)))
    ax.set_xticklabels(RADAR_LEADS)
    ax.set_yticks(np.arange(len(RADAR_KEYS)))
    ax.set_yticklabels(RADAR_KEYS)
    ax.set_xlabel('Time to hail onset (min)')
    ax.set_title('Radar robust difference', loc='center', fontsize=18, fontweight='bold', pad=6)
    ax.tick_params(length=0)
    return image

def latest_nwp_paired_effects(effect_df: pd.DataFrame) -> pd.DataFrame:
    sub = effect_df[effect_df['source'].eq('nwp') & effect_df['param'].isin(NWP_KEYS)].copy()
    if sub.empty:
        return pd.DataFrame()
    sub = sub.sort_values(['param', 'lead_min'], ascending=[True, False])
    return sub.groupby('param', as_index=False).first()

def choose_b_xlim(values: np.ndarray, ci_low: np.ndarray, ci_high: np.ndarray) -> Tuple[float, float]:
    if B_EFFECT_XLIM is not None:
        return B_EFFECT_XLIM
    candidates = np.concatenate([values, ci_low, ci_high])
    candidates = np.abs(candidates[np.isfinite(candidates)])
    if candidates.size == 0:
        bound = 1.0
    else:
        bound = max(0.5, float(np.max(candidates)) * 1.28)
    return (-bound, bound)

def plot_nwp_paired_effect(ax: plt.Axes, latest_nwp: pd.DataFrame) -> None:
    ordered = latest_nwp.set_index('param').reindex(NWP_KEYS).reset_index().dropna(subset=['paired_robust_effect'])
    if ordered.empty:
        ax.text(0.5, 0.5, 'No paired NWP effect', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        return
    values = ordered['paired_robust_effect'].to_numpy(float)
    ci_low = ordered['ci_low'].to_numpy(float)
    ci_high = ordered['ci_high'].to_numpy(float)
    y = np.arange(len(ordered))
    colors = np.where(values >= 0, LABEL_COLORS['Hail'], LABEL_COLORS['No hail'])
    left_error = values - ci_low
    right_error = ci_high - values
    xerr = np.vstack([left_error, right_error])
    valid_error = np.all(np.isfinite(xerr) & (xerr >= 0))
    if valid_error:
        ax.barh(y, values, xerr=xerr, color=colors, alpha=0.88, capsize=3, linewidth=0)
    else:
        ax.barh(y, values, color=colors, alpha=0.88, linewidth=0)
    ax.axvline(0, color='0.25', linewidth=0.8)
    ax.set_yticks(y)
    b_ylabels = [NWP_label.get(param, param) for param in ordered['param']]
    ax.set_yticklabels(b_ylabels, fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    xlim = choose_b_xlim(values, ci_low, ci_high)
    ax.set_xlim(-0.2, 0.8)
    ax.set_xlabel('Value')
    ax.set_title('NWP robust difference', loc='center', fontsize=18, fontweight='bold', pad=6)
    ax.spines[['top', 'right', 'left']].set_visible(False)
    for yi, value, lower, upper in zip(y, values, ci_low, ci_high):
        if value >= 0:
            anchor_x = max(value, upper) if np.isfinite(upper) else value
            offset_points = 8
            ha = 'left'
        else:
            anchor_x = min(value, lower) if np.isfinite(lower) else value
            offset_points = -8
            ha = 'right'
        ax.annotate(f'{value:+.2f}', xy=(anchor_x, yi), xytext=(offset_points, 0), textcoords='offset points', ha=ha, va='center', fontname='Arial', fontsize=14, clip_on=False)

def plot_radar_distribution(ax: plt.Axes, summary: pd.DataFrame, param: str) -> None:
    for label in LABEL_ORDER:
        current = summary[summary['label'].eq(label)].sort_values('lead_min')
        if current.empty:
            continue
        x = current['lead_min'].to_numpy(float)
        median = current['median'].to_numpy(float)
        low = current['q_low'].to_numpy(float)
        high = current['q_high'].to_numpy(float)
        color = LABEL_COLORS[label]
        ax.fill_between(x, low, high, color=color, alpha=0.14, linewidth=0)
        ax.plot(x, median, color=color, marker='o', markersize=2.2, linewidth=1.1, label=label)
    ax.axvline(0, color='0.35', linewidth=0.7, linestyle='--')
    ax.set_xlim(-63, 2)
    ax.set_xticks([-60, -48, -36, -24, -12, 0])
    ax.set_xlabel('Time to hail onset (min)')
    unit = PARAM_UNITS.get(param, '')
    ylabel = f'{RADAR_STAT.upper()} ({unit})' if unit else RADAR_STAT.upper()
    ax.set_ylabel(ylabel)
    if param in RADAR_YLIMS:
        ax.set_ylim(*RADAR_YLIMS[param])
    ax.set_title(param, loc='center', fontsize=18, fontweight='bold', pad=6)
    ax.spines[['top', 'right']].set_visible(False)

def plot_nwp_boxplot(ax: plt.Axes, case_values: pd.DataFrame, param: str) -> None:
    arrays = []
    labels_found = []
    for label in LABEL_ORDER:
        values = case_values.loc[case_values['label'].eq(label), 'value'].dropna().to_numpy(float)
        if values.size:
            arrays.append(values)
            labels_found.append(label)
    if not arrays:
        ax.text(0.5, 0.5, 'No NWP data', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        return
    positions = np.arange(1, len(arrays) + 1)
    box = ax.boxplot(arrays, positions=positions, widths=0.58, patch_artist=True, showfliers=False, whis=NWP_BOX_WHIS, medianprops={'color': 'black', 'linewidth': 1.0}, whiskerprops={'linewidth': 0.8}, capprops={'linewidth': 0.8})
    for patch, label in zip(box['boxes'], labels_found):
        color = LABEL_COLORS[label]
        patch.set_facecolor(color)
        patch.set_edgecolor(color)
        patch.set_alpha(0.28)
    rng = np.random.default_rng(20260728)
    max_points = 250
    for xpos, values, label in zip(positions, arrays, labels_found):
        if values.size > max_points:
            indices = rng.choice(values.size, size=max_points, replace=False)
            plot_values = values[indices]
        else:
            plot_values = values
        jitter = rng.normal(loc=0.0, scale=0.035, size=plot_values.size)
    short_labels = ['Hail' if label == 'Hail' else 'No hail' for label in labels_found]
    ax.set_xticks(positions)
    ax.set_xticklabels(short_labels)
    unit = PARAM_UNITS.get(param, '')
    ylabel = f'MAX ({unit})' if unit else 'MAX'
    ax.set_ylabel(ylabel)
    hail_values = case_values.loc[case_values['label'].eq('Hail'), 'value'].dropna().to_numpy(float)
    if hail_values.size > 0:
        ax.axhline(y=float(np.median(hail_values)), color=LABEL_COLORS['Hail'], linestyle='--', linewidth=1.4, alpha=0.9, zorder=0)
    if param in NWP_YLIMS:
        ax.set_ylim(*NWP_YLIMS[param])
    ax.set_title(nwp_title_map.get(param, param), loc='center', fontsize=18, fontweight='bold', pad=6)
    ax.spines[['top', 'right']].set_visible(False)

def make_figure(primary: pd.DataFrame, effect_df: pd.DataFrame) -> plt.Figure:
    configure_style()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig = plt.figure(figsize=(16, 15))
    grid = fig.add_gridspec(nrows=4, ncols=4, height_ratios=[1.25, 1.0, 1.0, 1.0], hspace=0.78, wspace=0.58)
    ax_a = fig.add_subplot(grid[0, 0:2])
    radar_effects = effect_df[effect_df['source'].eq('radar') & effect_df['param'].isin(RADAR_KEYS)].copy()
    if radar_effects.empty:
        ax_a.text(0.5, 0.5, 'No radar paired effect', ha='center', va='center', transform=ax_a.transAxes)
        ax_a.set_axis_off()
        heatmap = None
    else:
        heatmap = plot_paired_effect_heatmap(ax_a, radar_effects)
    add_panel_label(ax_a, 'A')
    if heatmap is not None:
        colorbar = fig.colorbar(heatmap, ax=ax_a, fraction=0.022, pad=0.018)
        colorbar.set_label('Value', family='Arial', size=14)
        colorbar.ax.tick_params(labelsize=14)
        for tick_label in colorbar.ax.get_yticklabels():
            tick_label.set_fontname('Arial')
            tick_label.set_fontsize(14)
    ax_b = fig.add_subplot(grid[0, 2:4])
    latest_nwp = latest_nwp_paired_effects(effect_df)
    plot_nwp_paired_effect(ax_b, latest_nwp)
    add_panel_label(ax_b, 'B')
    radar_axes = [fig.add_subplot(grid[1, column]) for column in range(4)] + [fig.add_subplot(grid[2, column]) for column in range(4)]
    panel_letters = iter(string.ascii_uppercase[2:])
    for ax, param in zip(radar_axes, RADAR_KEYS):
        summary = radar_distribution_summary(primary, param)
        plot_radar_distribution(ax, summary, param)
        add_panel_label(ax, next(panel_letters))
    nwp_axes = [fig.add_subplot(grid[3, column]) for column in range(4)]
    for ax, param in zip(nwp_axes, NWP_KEYS):
        case_values = nwp_case_level_values(primary, param)
        plot_nwp_boxplot(ax, case_values, param)
        add_panel_label(ax, next(panel_letters))
    all_axes = [ax_a, ax_b, *radar_axes, *nwp_axes]
    for current_ax in all_axes:
        apply_axis_font_style(current_ax)
    fig.subplots_adjust(left=0.055, right=0.985, top=0.975, bottom=0.055)
    fig.canvas.draw()
    handles, legend_labels = radar_axes[0].get_legend_handles_labels()
    if handles:
        first_radar_row = radar_axes[:4]
        row_left = min((current_ax.get_position().x0 for current_ax in first_radar_row))
        row_right = max((current_ax.get_position().x1 for current_ax in first_radar_row))
        row_top = max((current_ax.get_position().y1 for current_ax in first_radar_row))
        fig.legend(handles, legend_labels, loc='lower center', bbox_to_anchor=((row_left + row_right) / 2, row_top + RADAR_LEGEND_Y_OFFSET), bbox_transform=fig.transFigure, ncol=2, frameon=False, borderaxespad=0.0, handlelength=2.2, columnspacing=2.0, prop={'family': 'Arial', 'size': 14})
    quantile_tag = f'q{int(round(RADAR_Q_LOW * 1000)):03d}_{int(round(RADAR_Q_HIGH * 1000)):03d}'
    output_base = OUTPUT_DIR / f'Fig_pre_hail_statistics_paired_robust_{RADAR_STAT}_{NWP_STAT}_{quantile_tag}'
    output_pdf = Path(str(output_base) + '.pdf')
    output_tif = Path(str(output_base) + '.tif')
    output_png = Path(str(output_base) + '.png')
    return fig


In [ ]:
if __name__ == '__main__':
    print(f'Working directory: {Path.cwd()}')
    print(f'Statistics directory: {OUTPUT_DIR}')
    tidy = load_saved_dataframe(OUTPUT_DIR / 'tidy_stats')
    primary = select_primary_data(tidy)
    effect_df = load_paired_effect_table()
    figure = make_figure(primary, effect_df)
    plt.savefig(OUTPUT_ROOT / 'pre_hail_statistics.png', dpi=300, bbox_inches='tight')
